# Semana 10 — Exercícios de Aula: SQL Avançado e Performance

Notebook organizado **na ordem de aparição dos slides da Semana 10**.

Cada bloco indica o número do slide de origem e traz:

- **Exemplos** adaptados para o modelo de e-commerce usado em aula.
- **Exercícios** para os alunos resolverem.
- Consultas usando `fato_vendas`, `dim_cliente`, `dim_produto` e `dim_tempo`.

## Tabelas consideradas

### `fato_vendas`
Tabela fato com as vendas.

Colunas usadas nos exercícios:
- `venda_sk`
- `order_id`
- `cliente_sk`
- `produto_sk`
- `tempo_sk`
- `valor_produto`
- `valor_frete`
- `valor_total`

### `dim_cliente`
Dimensão de clientes.

Colunas usadas:
- `cliente_sk`
- `customer_city`
- `customer_state`

### `dim_produto`
Dimensão de produtos.

Colunas usadas:
- `produto_sk`
- `product_category_name_english`

### `dim_tempo`
Dimensão de tempo.

Colunas usadas:
- `tempo_sk`
- `data`
- `ano`
- `mes`
- `trimestre`
- `ano_mes`

## Preparação

Use este padrão para executar SQL no notebook, caso esteja usando DuckDB.

Se as tabelas já estiverem carregadas no ambiente, siga direto para os exemplos.

In [1]:
import duckdb
import pandas as pd


In [2]:
dim_cliente = pd.read_csv('../Dataset_schema/dim_cliente.csv')
dim_produto = pd.read_csv('../Dataset_schema/dim_produto.csv')
dim_tempo = pd.read_csv('../Dataset_schema/dim_tempo.csv')
fato_vendas = pd.read_csv('../Dataset_schema/fato_vendas.csv')

In [3]:
banco_dados = duckdb.connect()
banco_dados.register('dim_cliente', dim_cliente) # registra a dimensão cliente no banco de dados
banco_dados.register('dim_produto', dim_produto) # registra a dimensão produto no banco de dados
banco_dados.register('dim_tempo', dim_tempo) # registra a dimensão tempo no banco de dados
banco_dados.register('fato_vendas', fato_vendas) # registra a tabela fato vendas no banco de dados

---

# Slide 3 — Exemplo: Subconsultas

O slide apresenta subconsultas como consultas dentro de consultas.  
Aqui o exemplo foi adaptado para o nosso modelo de e-commerce.

## Exemplo — Vendas acima da média geral

Pergunta de negócio:

> Quais vendas tiveram valor total acima da média geral de vendas?

In [6]:
query = """
SELECT AVG(valor_total) AS media_geral
FROM fato_vendas
"""

banco_dados.sql(query).df()

,media_geral
0,139.929161


In [7]:
query = """
SELECT
    venda_sk,
    valor_total
FROM fato_vendas
WHERE valor_total > (
    SELECT AVG(valor_total)
    FROM fato_vendas
)
ORDER BY valor_total DESC;
"""

banco_dados.sql(query).df()

,venda_sk,valor_total
0,3482,6929.31
1,109785,6922.21
2,105496,6726.66
3,72721,4950.34
4,10998,4764.34
...,...,...
32858,62148,139.93
32859,62149,139.93
32860,93983,139.93
32861,81520,139.93


---

# Slide 4 — Exercícios: Subconsultas na prática

Resolva usando subconsultas.

## Slide 4 — Fácil: Vendas acima da média geral

Liste as vendas com `valor_total` acima da média geral.

Retorne:
- `venda_sk`
- `order_id`
- `valor_total`

Ordene por `valor_total` decrescente.

In [5]:
# Resposta do aluno

query = """
SELECT
	venda_sk,
    order_id,
    valor_total
FROM fato_vendas
WHERE valor_total > (
	SELECT
		AVG(valor_total)
    FROM fato_vendas
    )
ORDER BY valor_total DESC
"""

banco_dados.sql(query).df()

,venda_sk,order_id,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,4764.34
...,...,...,...
32858,62148,911c0c489bf4354f38fc64491075e8ba,139.93
32859,62149,911c0c489bf4354f38fc64491075e8ba,139.93
32860,93983,d9f134a34fb4b3cfd95550b4785e8d05,139.93
32861,81520,bd3b1eb371dd6c8ca77447b363de08cd,139.93


## Slide 4 — Médio — Estados com volume acima da média

Calcule a quantidade de pedidos por estado e liste apenas os estados com quantidade de pedidos acima da média dos estados.

Use:
- `fato_vendas`
- `dim_cliente`

Retorne:
- `estado`
- `qtd_pedidos`

In [18]:
# Resposta do aluno

query = """
SELECT
	c.customer_state AS estado,
	COUNT(f.venda_sk) AS qtd_pedidos
FROM fato_vendas f
JOIN dim_cliente c
	ON f.cliente_sk = c.cliente_sk
GROUP BY c.customer_state
HAVING COUNT(*) > (
	SELECT
		AVG(total)
    FROM (
		SELECT
			COUNT(*) AS total
        FROM fato_vendas
        GROUP BY cliente_sk
    )
)
"""

banco_dados.sql(query).df()

,estado,qtd_pedidos
0,PE,1746
1,MG,12916
2,MA,800
3,DF,2355
4,SC,4097
5,RS,6134
6,GO,2277
7,AC,91
8,PA,1054
9,ES,2225


## Slide 4 — Difícil — Vendas das 3 categorias com maior receita

Retorne as vendas pertencentes às 3 categorias com maior receita total.

Use uma subconsulta para descobrir as 3 categorias.

Retorne:
- `venda_sk`
- `order_id`
- `categoria`
- `valor_total`

Dica:
Use esta subconsulta:  
WHERE v.produto_sk IN (

-- SUB-CONSULTA: Agrupa por produto e conta a quantidade de linhas (pedidos)  
-- Ordena do mais vendido para o menos vendido e limita aos 3 primeiros  

SELECT produto_sk  
FROM fato_vendas  
GROUP BY produto_sk  
ORDER BY COUNT(order_id) DESC   
LIMIT 3). 


In [ ]:
# Resposta do aluno
query = """
SELECT
	f.venda_sk,
    f.order_id,
    p.product_category_name AS categoria,
    f.valor_total
FROM fato_vendas f
JOIN dim_produto p
	ON f.produto_sk = p.produto_sk
WHERE f.produto_sk IN (

-- SUB-CONSULTA: Agrupa por produto e conta a qtidade de linhas (pedidos)
-- Ordena do mais vendido para o menos vendido e limita aos 3 primeiros

	SELECT 
    	produto_sk
    FROM fato_vendas
    GROUP BY produto_sk
    ORDER BY 
		COUNT(order_id) DESC
    LIMIT 3
)

ORDER BY
	p.product_category_name,
    f.valor_total DESC
"""

banco_dados.sql(query).df().round(2)


,venda_sk,order_id,categoria,valor_total
0,26174,3ce7418ccb9af136b4937f069601ab2f,cama_mesa_banho,137.23
1,26723,3e2b45761c1280a1c1165c83d3292c43,cama_mesa_banho,132.94
2,85118,c5883e5be24c4f0627d36a56ed34d190,cama_mesa_banho,132.94
3,87947,cc2e6cf82767599713d657cd5cf8b300,cama_mesa_banho,132.94
4,6271,0e8594074c3143aa6b56a3ec70a46e12,cama_mesa_banho,132.94
...,...,...,...,...
1476,4423,0a2f51f07c7b99722f843490cd20ec09,moveis_decoracao,69.90
1477,43605,6554129577eff36836f1a93894b64f42,moveis_decoracao,69.90
1478,22016,336849260da128bf88be93e5bd4d15be,moveis_decoracao,69.90
1479,79207,b811c8ca07716586e80c0bc4f2a6e39f,moveis_decoracao,69.90


---

# Slide 6 — Exemplo: Common Table Expression, CTE

O slide apresenta CTE como uma consulta nomeada criada com `WITH`.

## Exemplo — Vendas de SP como CTE

Pergunta de negócio:

> Como criar uma etapa intermediária com as vendas de clientes de SP?

In [7]:
query = """
WITH vendas_sp AS (
    SELECT
        f.venda_sk,
        f.order_id,
        f.valor_total,
        c.customer_state AS estado
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    WHERE c.customer_state = 'SP'
)
SELECT *
FROM vendas_sp
ORDER BY valor_total DESC
LIMIT 20;
"""

banco_dados.sql(query).df()

,venda_sk,order_id,valor_total,estado
0,105496,f5136e38d1a14a4dbd87dff67da82701,6726.66,SP
1,10998,199af31afc78c699f0dbf71fb178d4d4,4764.34,SP
2,28537,426a9742b533fc6fed17d1fd6d143d7e,4513.32,SP
3,67630,9de73f3e6157169ad6c32b9f313c7dcb,4034.44,SP
4,57838,86c4eab1571921a6a6e248ed312f5a5a,4016.91,SP
5,25064,3a4b013e014723cc38c9faa8ffdc6387,3526.46,SP
6,28078,4160b6447ad70e6d73f7221dd970bbe9,3076.13,SP
7,81498,bd2fef198085db0b586b9c71aa2d35da,3024.08,SP
8,38030,586992f50ed97898737b07970376d19c,3016.01,SP
9,91511,d3fc45461660252856d603ed31eb4e77,2938.17,SP


---

# Slide 7 — Exemplo: Sintaxe de CTE e múltiplas CTEs

O slide mostra a estrutura básica de uma CTE e o uso de múltiplas CTEs.

In [12]:
# Exemplo do Slide 7 com múltiplas CTEs

query = """
WITH receita_por_categoria AS (
    SELECT
        p.product_category_name_english AS categoria,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    GROUP BY p.product_category_name_english
),
categorias_relevantes AS (
    SELECT
        categoria,
        receita_total
    FROM receita_por_categoria
    WHERE receita_total > 500000
)
SELECT *
FROM categorias_relevantes
ORDER BY receita_total DESC;
"""

banco_dados.sql(query).df()

,categoria,receita_total
0,health_beauty,1412089.53
1,watches_gifts,1264333.12
2,bed_bath_table,1225209.26
3,sports_leisure,1118256.91
4,computers_accessories,1032723.77
5,furniture_decor,880329.92
6,housewares,758392.25
7,cool_stuff,691680.89
8,auto,669454.75
9,garden_tools,567145.68


---

# Slide 8 — Exercícios: CTEs na prática

Reescreva consultas usando `WITH` para deixar a lógica mais clara.

## Slide 8 — Exercício 1 — Vendas por estado

Crie uma CTE chamada `vendas_por_estado` com a quantidade de pedidos e a receita total por estado.

In [12]:
# Resposta do aluno

query = """
WITH vendas_por_estado AS (
	SELECT
		c.customer_state AS estado,
		COUNT(f.order_id) AS qtd_pedidos,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    JOIN dim_cliente c
		ON f.cliente_sk = c.cliente_sk
	GROUP BY estado
)
SELECT *
FROM vendas_por_estado
ORDER BY receita_total DESC
"""

banco_dados.sql(query).df()

,estado,qtd_pedidos,receita_total
0,SP,46448,5769703.15
1,RJ,14143,2055401.57
2,MG,12916,1818891.67
3,RS,6134,861472.79
4,PR,5649,781708.80
5,SC,4097,595127.78
6,BA,3683,591137.81
7,DF,2355,346123.35
8,GO,2277,334212.35
9,ES,2225,317657.93


## Slide 8 — Exercício 2 — Análise por categoria
Crie uma CTE chamada resumo_categorias para calcular a quantidade de vendas, a receita total e o ticket médio por categoria de produto. Depois, na consulta final, mostre apenas as categorias com receita total acima de R$ 100.000.

In [15]:
# Resposta do aluno

query = """
WITH resumo_categoria AS (
	SELECT
		p.product_category_name AS categoria,
		COUNT(f.venda_sk) AS qtd_vendas,
        SUM(f.valor_total) AS receita_total,
        AVG(f.valor_total) AS ticket_medio
    FROM fato_vendas f
    JOIN dim_produto p
		ON f.produto_sk = p.produto_sk
    GROUP BY categoria
)
SELECT *
FROM resumo_categoria
WHERE receita_total > 100000
ORDER BY receita_total DESC
"""

banco_dados.sql(query).df().round(2)

,categoria,qtd_vendas,receita_total,ticket_medio
0,beleza_saude,9465,1412089.53,149.19
1,relogios_presentes,5859,1264333.12,215.79
2,cama_mesa_banho,10953,1225209.26,111.86
3,esporte_lazer,8431,1118256.91,132.64
4,informatica_acessorios,7644,1032723.77,135.10
5,moveis_decoracao,8160,880329.92,107.88
6,utilidades_domesticas,6795,758392.25,111.61
7,cool_stuff,3718,691680.89,186.04
8,automotivo,4140,669454.75,161.70
9,ferramentas_jardim,4268,567145.68,132.88


## Slide 8 — Exercício 3 — Top categorias por estado

Crie CTEs para descobrir quais são as categorias de produto com maior receita em cada estado.
Primeiro, calcule a receita total por estado e categoria. Depois, filtre apenas combinações com receita acima de R$ 50.000 e ordene o resultado por estado e receita.

In [ ]:
# Resposta do aluno

query = """
WITH receita_por_estado_categoria AS (
	SELECT
		c.customer_state AS estado,
        p.product_category_name AS categoria,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    JOIN dim_cliente c
    	ON f.cliente_sk = c.cliente_sk
    JOIN dim_produto p
		ON f.produto_sk = p.produto_sk
    GROUP BY estado, categoria
)

SELECT *
FROM receita_por_estado_categoria
WHERE receita_total > 50000
ORDER BY estado, receita_total DESC
"""

banco_dados.sql(query).df().round(2)

,estado,categoria,receita_total
0,BA,beleza_saude,58620.07
1,BA,relogios_presentes,50254.70
2,MG,beleza_saude,175007.52
3,MG,cama_mesa_banho,156483.10
4,MG,relogios_presentes,132311.65
...,...,...,...
61,SP,construcao_ferramentas_construcao,66397.74
62,SP,consoles_games,65789.05
63,SP,fashion_bolsas_e_acessorios,65487.98
64,SP,pcs,64371.71


---

# Slide 11 — Conceito: Window Functions

O slide compara `GROUP BY` e `OVER()`.

- `GROUP BY` resume os dados e reduz a quantidade de linhas.
- `OVER()` calcula uma métrica sem apagar o detalhe das linhas.

A partir daqui, os exemplos usam Window Functions.

---

# Slide 12 — Exemplo: Anatomia de uma Window Function

O slide apresenta `PARTITION BY`, `ORDER BY` e a função `ROW_NUMBER()`.

In [24]:
# Exemplo do Slide 12 adaptado

query = """
SELECT
    f.venda_sk,
    f.order_id,
    p.product_category_name AS categoria,
    f.valor_total,
    ROW_NUMBER() OVER (
        PARTITION BY p.product_category_name
        ORDER BY f.valor_total DESC
    ) AS ranking_venda_categoria
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
WHERE p.product_category_name IS NOT NULL
ORDER BY
    p.product_category_name,
    ranking_venda_categoria;
"""

banco_dados.sql(query).df()

,venda_sk,order_id,categoria,valor_total,ranking_venda_categoria
0,93047,d7a2c0c1ff66b314f3bf166fb4157fd4,agro_industria_e_comercio,3184.55,1
1,68308,9f5abaa01d419a15e448f089b1cf4b94,agro_industria_e_comercio,2467.33,2
2,83578,c205abf27ff07fddd3fa9a67037138e5,agro_industria_e_comercio,2315.83,3
3,10860,193d998b23a996345ca322516b2d5af3,agro_industria_e_comercio,1978.18,4
4,15260,23aee6ac2f5a0a3056aa62c615974c0c,agro_industria_e_comercio,1518.55,5
...,...,...,...,...,...
108655,80364,baa821bd1dbd5c6d31c9b1c68f760b8b,utilidades_domesticas,11.85,6791
108656,61575,8fbcb92faf1aa60361f61ed7ae721a7e,utilidades_domesticas,6.08,6792
108657,61573,8fbcb92faf1aa60361f61ed7ae721a7e,utilidades_domesticas,6.08,6793
108658,61577,8fbcb92faf1aa60361f61ed7ae721a7e,utilidades_domesticas,6.08,6794


---

# Slide 13 — Exemplo: ROW_NUMBER()

`ROW_NUMBER()` cria uma numeração sequencial para cada linha dentro da partição.

In [29]:
# Exemplo do Slide 13 adaptado

query = """
WITH ranking_vendas_categoria AS (
    SELECT
        f.venda_sk,
        f.order_id,
        p.product_category_name AS categoria,
        f.valor_total,
        ROW_NUMBER() OVER (
            PARTITION BY p.product_category_name
            ORDER BY f.valor_total DESC
        ) AS posicao
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
)

SELECT
    venda_sk,
    order_id,
    categoria,
    valor_total,
    posicao
FROM ranking_vendas_categoria
WHERE posicao = 1
ORDER BY valor_total DESC;
"""

banco_dados.sql(query).df()

,venda_sk,order_id,categoria,valor_total,posicao
0,3482,0812eb902a67711a1cb742b3cdaa65ae,utilidades_domesticas,6929.31,1
1,109785,fefacc66af859508bf1a7934eab1e97f,pcs,6922.21,1
2,105496,f5136e38d1a14a4dbd87dff67da82701,artes,6726.66,1
3,72721,a96610ab360d42a2e5335a3998b4718a,eletroportateis,4950.34,1
4,28537,426a9742b533fc6fed17d1fd6d143d7e,instrumentos_musicais,4513.32,1
...,...,...,...,...,...
68,46240,6b7c8bd6aa5cfe7f791eb8caa3da1701,fashion_underwear_e_moda_praia,208.09,1
69,52169,796033ffde6ba06e0abe387e36ee8fe0,fraldas_higiene,158.97,1
70,102718,ee86b68eb9222b0cad7da50f4f758a35,fashion_roupa_infanto_juvenil,124.52,1
71,54567,7ed69fbc79fbda50e09caa9c127026e5,cds_dvds_musicais,117.58,1


---

# Slide 14 — Exemplo: RANK() e DENSE_RANK()

O slide mostra a diferença entre rankings com empate.

In [31]:
# Exemplo do Slide 14 adaptado

query = """
WITH receita_por_categoria AS (
    SELECT
        p.product_category_name AS categoria,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY
        p.product_category_name
)

SELECT
    categoria,
    ROUND(receita_total, 2) AS receita_total,

    RANK() OVER (
        ORDER BY receita_total DESC
    ) AS rank_categoria,

    DENSE_RANK() OVER (
        ORDER BY receita_total DESC
    ) AS dense_rank_categoria

FROM receita_por_categoria
ORDER BY receita_total DESC;
"""

banco_dados.sql(query).df()

,categoria,receita_total,rank_categoria,dense_rank_categoria
0,beleza_saude,1412089.53,1,1
1,relogios_presentes,1264333.12,2,2
2,cama_mesa_banho,1225209.26,3,3
3,esporte_lazer,1118256.91,4,4
4,informatica_acessorios,1032723.77,5,5
...,...,...,...,...
68,pc_gamer,1430.10,69,69
69,casa_conforto_2,1170.58,70,70
70,cds_dvds_musicais,954.99,71,71
71,fashion_roupa_infanto_juvenil,598.67,72,72


---

# Slide 15 — Exemplo: SUM() OVER()

O slide apresenta acumulados, médias móveis e percentual do total.

## Slide 15 — Exemplo 1: receita acumulada por estado

In [40]:
# Exemplo do Slide 15

query = """
SELECT
    p.product_category_name AS categoria,
    f.order_id,
    f.valor_total,

    SUM(f.valor_total) OVER (
        PARTITION BY p.product_category_name
    ) AS total_categoria,

    100.0 * f.valor_total / SUM(f.valor_total) OVER (
        PARTITION BY p.product_category_name
    ) AS percentual_na_categoria

FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
WHERE p.product_category_name IS NOT NULL;

"""

banco_dados.sql(query).df()

,categoria,order_id,valor_total,total_categoria,percentual_na_categoria
0,esporte_lazer,2b1e4967ed6ca5b1edd58b616b4da7e8,126.37,1118256.91,0.011301
1,esporte_lazer,2b2e47d32f64348664d0b1c367308a67,114.44,1118256.91,0.010234
2,esporte_lazer,2b36fe8c73edb9af80b2d3e73b46122a,111.19,1118256.91,0.009943
3,esporte_lazer,2b36fe8c73edb9af80b2d3e73b46122a,111.19,1118256.91,0.009943
4,esporte_lazer,2b3ec451746cbc17ebae5c0b8fc74406,166.29,1118256.91,0.014870
...,...,...,...,...,...
108655,beleza_saude,5e8752fb422500b68d03de8db7555dc6,185.62,1412089.53,0.013145
108656,beleza_saude,5e8dbd59abee490ba3f319a92089c668,88.51,1412089.53,0.006268
108657,beleza_saude,5e8dbd76385140aefb062d9ed31c7a75,667.99,1412089.53,0.047305
108658,beleza_saude,5e8dbed878ecb57caf14b67e8f85ca34,69.45,1412089.53,0.004918


---

# Slide 16 — Boas práticas: Otimizando consultas com Window Functions

Antes de usar Window Function em uma tabela muito grande:

1. Filtre os dados antes com `WHERE` ou CTE.
2. Calcule apenas as colunas necessárias.
3. Evite `PARTITION BY` sem necessidade.
4. Observe colunas usadas em `PARTITION BY` e `ORDER BY`.

---

# Slide 17 — Exercícios

Aplique Window Functions para resolver os problemas abaixo.

## Slide 17 — Exercício 1 — Pedido mais caro de cada estado

Use `ROW_NUMBER()` para encontrar o pedido de maior valor em cada estado.

In [47]:
# Resposta do aluno
query = """
WITH ranking_pedidos_estado AS (
	SELECT 
		c.customer_state AS estado,
        f.order_id,
        f.valor_total,
        
        ROW_NUMBER() OVER(
			PARTITION BY c.customer_state
            ORDER BY f.valor_total DESC
        ) AS posicao
        
    FROM fato_vendas f
    JOIN dim_cliente c
		ON f.cliente_sk = c.cliente_sk
)

SELECT
	estado,
    order_id,
    ROUND(valor_total, 2) AS valor_total,
    posicao
FROM ranking_pedidos_estado
WHERE posicao = 1
ORDER BY valor_total DESC

"""

banco_dados.sql(query).df()


,estado,order_id,valor_total,posicao
0,MS,0812eb902a67711a1cb742b3cdaa65ae,6929.31,1
1,ES,fefacc66af859508bf1a7934eab1e97f,6922.21,1
2,SP,f5136e38d1a14a4dbd87dff67da82701,6726.66,1
3,RJ,a96610ab360d42a2e5335a3998b4718a,4950.34,1
4,PB,8dbc85d1447242f3b127dda390d56e19,4681.78,1
5,DF,80dfedb6d17bf23539beeef3c768f4d7,4194.76,1
6,MG,68101694e5c5dc7330c91e1bbc36214f,4175.26,1
7,PA,9a3966c23190dbdbaabed08e8429c006,4042.74,1
8,PE,d3f66901a6743e15f9311547cc623b91,3792.59,1
9,RS,f398a143c0fe171d965db2096cf064cf,3297.40,1


## Slide 17 — Exercício 2 — Ranking de estados por total vendido

Calcule a receita total por estado e use 'RANK()' para ranquear os estados com maior faturamento.

In [61]:
# Resposta do aluno
query = """
WITH receita_total_estado AS (
	SELECT
		c.customer_state AS estado,
        COUNT(DISTINCT order_id) AS qtd_pedidos,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    JOIN dim_cliente c
		ON f.cliente_sk = c.cliente_sk
    GROUP BY estado
)

SELECT
	estado,
    qtd_pedidos,
    ROUND(receita_total, 2) AS receita_total,
    
    RANK() OVER(
        ORDER BY receita_total DESC
    ) AS ranking_estado
    
FROM receita_total_estado
ORDER BY receita_total DESC
"""

banco_dados.sql(query).df()


,estado,qtd_pedidos,receita_total,ranking_estado
0,SP,40501,5769703.15,1
1,RJ,12350,2055401.57,2
2,MG,11354,1818891.67,3
3,RS,5345,861472.79,4
4,PR,4923,781708.80,5
5,SC,3546,595127.78,6
6,BA,3256,591137.81,7
7,DF,2080,346123.35,8
8,GO,1957,334212.35,9
9,ES,1995,317657.93,10


## Slide 17 — Exercício 3 — Receita acumulada mensal por estado

Calcule a receita mensal por estado e depois o acumulado mês a mês. Use SUM() OVER()

In [ ]:
#Resposta do aluno

query = """
-- coloque sua resposta aqui
"""

banco_dados.sql(query).df()


### GABARITO EXERCÍCIOS SLIDE 17

In [ ]:
# Gabarito do exercício 1 slide 17

query = """
-- coloque sua resposta aqui

WITH ranking_pedidos_estado AS (
    SELECT
        c.customer_state AS estado,
        f.order_id,
        f.valor_total,
        
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_state
            ORDER BY f.valor_total DESC
        ) AS posicao
        
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
)

SELECT
    estado,
    order_id,
    ROUND(valor_total, 2) AS valor_total
FROM ranking_pedidos_estado
WHERE posicao = 1
ORDER BY valor_total DESC;


"""

banco_dados.sql(query).df()

,estado,order_id,valor_total
0,MS,0812eb902a67711a1cb742b3cdaa65ae,6929.31
1,ES,fefacc66af859508bf1a7934eab1e97f,6922.21
2,SP,f5136e38d1a14a4dbd87dff67da82701,6726.66
3,RJ,a96610ab360d42a2e5335a3998b4718a,4950.34
4,PB,8dbc85d1447242f3b127dda390d56e19,4681.78
5,DF,80dfedb6d17bf23539beeef3c768f4d7,4194.76
6,MG,68101694e5c5dc7330c91e1bbc36214f,4175.26
7,PA,9a3966c23190dbdbaabed08e8429c006,4042.74
8,PE,d3f66901a6743e15f9311547cc623b91,3792.59
9,RS,f398a143c0fe171d965db2096cf064cf,3297.40


In [33]:
# Gabarito do exercício 2 slide 17

query = """
WITH receita_por_estado AS (
    SELECT
        c.customer_state AS estado,
        SUM(f.valor_total) AS receita_total,
        COUNT(DISTINCT f.order_id) AS qtd_pedidos
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    GROUP BY
        c.customer_state
)

SELECT
    estado,
    qtd_pedidos,
    ROUND(receita_total, 2) AS receita_total,
    RANK() OVER (
        ORDER BY receita_total DESC
    ) AS ranking_estado
FROM receita_por_estado
ORDER BY ranking_estado;
"""

banco_dados.sql(query).df()

,estado,qtd_pedidos,receita_total,ranking_estado
0,SP,40501,5769703.15,1
1,RJ,12350,2055401.57,2
2,MG,11354,1818891.67,3
3,RS,5345,861472.79,4
4,PR,4923,781708.80,5
5,SC,3546,595127.78,6
6,BA,3256,591137.81,7
7,DF,2080,346123.35,8
8,GO,1957,334212.35,9
9,ES,1995,317657.93,10


In [32]:
# Gabarito do exercício 3 slide 17

query = """
WITH receita_mensal_estado AS (
    SELECT
        c.customer_state AS estado,
        t.ano,
        t.mes,
        t.ano_mes,
        SUM(f.valor_total) AS receita_mes
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY
        c.customer_state,
        t.ano,
        t.mes,
        t.ano_mes
)

SELECT
    estado,
    ano,
    mes,
    ano_mes,
    receita_mes,
    SUM(receita_mes) OVER (
            PARTITION BY estado
            ORDER BY ano, mes
        ) AS receita_acumulada_estado
FROM receita_mensal_estado
ORDER BY
    estado,
    ano,
    mes;
"""

banco_dados.sql(query).df()

,estado,ano,mes,ano_mes,receita_mes,receita_acumulada_estado
0,AC,2017,1,2017-01,723.14,723.14
1,AC,2017,2,2017-02,597.40,1320.54
2,AC,2017,3,2017-03,530.18,1850.72
3,AC,2017,4,2017-04,1351.51,3202.23
4,AC,2017,5,2017-05,2371.73,5573.96
...,...,...,...,...,...,...
551,TO,2018,4,2018-04,5370.51,45245.41
552,TO,2018,5,2018-05,3314.33,48559.74
553,TO,2018,6,2018-06,4987.03,53546.77
554,TO,2018,7,2018-07,3795.09,57341.86


---

# Slide 20 — Exemplo: Índices e Otimização

O slide mostra uma consulta típica que pode se beneficiar de índice.

A consulta original foi corrigida para usar `INNER JOIN` com `dim_cliente` e `dim_tempo`.

In [45]:
# Exemplo do Slide 20

query = """
SELECT
    f.venda_sk,
    f.order_id,
    f.cliente_sk,
    f.produto_sk,
    f.tempo_sk,
    f.valor_total,
    c.customer_state AS estado,
    t.data AS data_venda
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
INNER JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
WHERE c.customer_state = 'SP'
  AND t.data >= '2018-01-01'
ORDER BY
    t.data,
    f.valor_total DESC;
"""

banco_dados.sql(query).df()

,venda_sk,order_id,cliente_sk,produto_sk,tempo_sk,valor_total,estado,data_venda
0,46474,6c06e4f4212e106eaa318b8330944588,5772,14458,319,234.46,SP,2018-01-01
1,9472,162e915bc5d7c6b2de7eca743a65b033,12824,18453,319,217.80,SP,2018-01-01
2,77205,b37a7be6fbd41461b87deafd8b902cc6,54842,24075,319,210.60,SP,2018-01-01
3,30089,45e4d1d16d017409456807846ea959e9,56917,20058,319,153.27,SP,2018-01-01
4,22042,33719438c660a7964c608082a59a58c6,90293,31293,319,141.40,SP,2018-01-01
...,...,...,...,...,...,...,...,...
26820,23002,35a972d7f8436f405b56e36add1a7140,67282,15035,595,93.75,SP,2018-08-29
26821,86318,c84d88553f9878bf2c7ecda2eb211ece,94863,4858,595,74.21,SP,2018-08-29
26822,35370,52018484704db3661b98ce838612b507,62463,17792,595,73.10,SP,2018-08-29
26823,1692,03ef5dedbe7492bdae72eec50764c43f,49886,10555,595,33.23,SP,2018-08-29


---

# Slide 22 — Exemplo: EXPLAIN antes e depois do índice

Use `EXPLAIN` para observar se o banco faz leitura sequencial ou usa algum índice.

In [50]:
# Exemplo do Slide 22 adaptado

query = """
EXPLAIN
SELECT *
FROM fato_vendas
WHERE cliente_sk = 5000;
"""

resultado = banco_dados.sql(query).df()

print(resultado["explain_value"][0])

┌───────────────────────────┐
│           FILTER          │
│    ────────────────────   │
│    (cliente_sk = 5000)    │
│                           │
│        ~22,039 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│        PANDAS_SCAN        │
│    ────────────────────   │
│   Function: PANDAS_SCAN   │
│                           │
│        Projections:       │
│          venda_sk         │
│          order_id         │
│       order_item_id       │
│         cliente_sk        │
│         produto_sk        │
│          tempo_sk         │
│       valor_produto       │
│        valor_frete        │
│        valor_total        │
│                           │
│       ~110,197 rows       │
└───────────────────────────┘



In [54]:
banco_dados.sql("""
CREATE TABLE fato_vendas_base AS
SELECT *
FROM fato_vendas;
""")

In [56]:
banco_dados.sql("""
CREATE INDEX idx_fato_vendas_cliente_sk
ON fato_vendas_base (cliente_sk);
""")

In [83]:


query = """
EXPLAIN
SELECT
    venda_sk,
    order_id,
    cliente_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000;
"""

resultado = banco_dados.sql(query).df()

print(resultado["explain_value"][0])
banco_dados.sql(query).df()

┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│          venda_sk         │
│          order_id         │
│         cliente_sk        │
│        valor_total        │
│                           │
│          ~2 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         SEQ_SCAN          │
│    ────────────────────   │
│           Table:          │
│      fato_vendas_base     │
│                           │
│   Type: Sequential Scan   │
│                           │
│        Projections:       │
│         cliente_sk        │
│          venda_sk         │
│          order_id         │
│        valor_total        │
│                           │
│          Filters:         │
│      cliente_sk=5000      │
│                           │
│          ~2 rows          │
└───────────────────────────┘



,explain_key,explain_value
0,physical_plan,┌───────────────────────────┐\n│ PROJE...


### Tranformando os dataframes em tabelas físicas dentro do DuckDB (antes estavamos usando o register que faz o DuckDB como uma fonte externa, porém a título de didática para indices precisamos fazer essa transformação)

In [69]:
# Transformando os DataFrames registrados em tabelas físicas no DuckDB
banco_dados.sql("DROP TABLE IF EXISTS fato_vendas_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_cliente_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_produto_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_tempo_base")


banco_dados.sql("""
CREATE TABLE fato_vendas_base AS
SELECT *
FROM fato_vendas
""")

banco_dados.sql("""
CREATE TABLE dim_cliente_base AS
SELECT *
FROM dim_cliente
""")

banco_dados.sql("""
CREATE TABLE dim_produto_base AS
SELECT *
FROM dim_produto
""")

banco_dados.sql("""
CREATE TABLE dim_tempo_base AS
SELECT *
FROM dim_tempo
""")

---

# Slide 24 — Exemplos: Criando índices

O slide apresenta índices simples e compostos.  
Abaixo estão exemplos adaptados para as tabelas do DW.

In [70]:


banco_dados.sql("""
-- Índice simples em chave de tempo
-- Ajuda em consultas que filtram vendas por período
CREATE INDEX idx_dim_tempo_data
ON dim_tempo_base (data);
""")

banco_dados.sql("""
-- Índice em coluna categórica da dimensão cliente
-- Ajuda em consultas que filtram clientes por estado
CREATE INDEX idx_dim_cliente_estado
ON dim_cliente_base (customer_state);
""")

banco_dados.sql("""
-- Índice composto
-- Ajuda em consultas que filtram por cliente e produto juntos
CREATE INDEX idx_fato_vendas_cliente_produto
ON fato_vendas_base (cliente_sk, produto_sk);
""")

In [74]:
query ="""
SELECT
    f.venda_sk,
    f.order_id,
    f.valor_total,
    t.data
FROM fato_vendas_base f
INNER JOIN dim_tempo_base t
    ON f.tempo_sk = t.tempo_sk
WHERE t.data >= '2018-01-01';
"""

banco_dados.sql(query).df()

,venda_sk,order_id,valor_total,data
0,3,000229ec398224ef6ca0657da4fc703e,216.87,2018-01-14
1,4,00024acbcdf0a6daa1e931b038114c75,25.78,2018-08-08
2,8,000576fe39319847cbb9d288c5617fa6,880.75,2018-07-04
3,9,0005a1a1728c9d785b8e2b08b904576c,157.60,2018-03-19
4,10,0005f50442cb953dcd1d21e1fb923495,65.39,2018-07-02
...,...,...,...,...
60319,110187,fffb2ef8874127f75b52b643880fd7e0,39.96,2018-03-30
60320,110192,fffbee3b5462987e66fb49b1c5411df2,139.88,2018-06-19
60321,110193,fffc94f6ce00a00581880bf54a75a037,343.40,2018-04-23
60322,110194,fffcd46ef2263f404302a634eb57f7eb,386.53,2018-07-14


In [75]:

query = """
SELECT
    cliente_sk,
    customer_id,
    customer_state
FROM dim_cliente_base
WHERE customer_state = 'SP';"""

banco_dados.sql(query).df()

,cliente_sk,customer_id,customer_state
0,1,06b8999e2fba1a1fbc88172c00ba8bc7,SP
1,2,18955e83d337fd6b2def6b18a428ac77,SP
2,3,4e7b3e00288586ebd08712fdd0374a03,SP
3,4,b2b6027bc5c5109e529d4dc6358b12c3,SP
4,5,4f2d8ab171c80ec8364f7c12e35b23ad,SP
...,...,...,...
41741,99433,f255d679c7c86c24ef4861320d5b7675,SP
41742,99435,f5a0b560f9e9427792a88bec97710212,SP
41743,99437,17ddf5dd5d51696bb3d7c6291687be6f,SP
41744,99438,e7b71a9017aa05c9a7fd292d714858e8,SP


In [84]:

query = """

SELECT
    venda_sk,
    order_id,
    cliente_sk,
    produto_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000
  AND produto_sk = 15420;

  """

banco_dados.sql(query).df()

,venda_sk,order_id,cliente_sk,produto_sk,valor_total
0,71606,a6dfab363a9e239287c4b1e3c275fb93,5000,15420,336.56


---

# Slide 25 — Exercícios: Índices

Identifique a coluna usada no filtro e crie o índice adequado.

Considere a transformação necessário no banco usada antes do inicio da aula aqui no VSCODE.  
Rode as cédulas abaixo

In [90]:
# Transformando os DataFrames registrados em tabelas físicas no DuckDB

banco_dados.sql("DROP TABLE IF EXISTS fato_vendas_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_cliente_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_produto_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_tempo_base")

banco_dados.sql("""
CREATE TABLE fato_vendas_base AS
SELECT *
FROM fato_vendas
""")

banco_dados.sql("""
CREATE TABLE dim_cliente_base AS
SELECT *
FROM dim_cliente
""")

banco_dados.sql("""
CREATE TABLE dim_produto_base AS
SELECT *
FROM dim_produto
""")

banco_dados.sql("""
CREATE TABLE dim_tempo_base AS
SELECT *
FROM dim_tempo
""")

## Slide 25 — Exercício 1 — Índice para filtro por estado

Consulta base:

```sql
SELECT
    cliente_sk,
    customer_id,
    customer_state
FROM dim_cliente_base
WHERE customer_state = 'SP'

```

Crie um índice adequado.

In [ ]:
# Resposta do aluno

query = """
CREATE INDEX -- resposta aqui

ON dim_cliente_base (resposta_aqui);
"""

## Slide 26 — Exercício 2 — Índice para consulta com JOIN e filtro por data

Como `customer_state` está na dimensão cliente, pense nos índices necessários para uma consulta com `JOIN`.

Consulta base:

```sql
SELECT
    f.venda_sk,
    f.order_id,
    f.valor_total,
    t.data
FROM fato_vendas_base f
INNER JOIN dim_tempo_base t
    ON f.tempo_sk = t.tempo_sk
WHERE t.data >= '2018-01-01';

```

In [111]:
# Resposta do aluno

query = """
CREATE INDEX -- resposta aqui
ON dim_tempo_base (-- resposta aqui);
"""


## Slide 27 — Exercício 3 — Índice composto para filtro combinado

Como `customer_state` está na dimensão cliente, pense nos índices necessários para uma consulta com `JOIN`.

Consulta base:

```sql
SELECT
    venda_sk,
    order_id,
    cliente_sk,
    produto_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000
  AND produto_sk = 15420;

```

In [ ]:
# Resposta do aluno

query = """
CREATE INDEX --resposta aqui
ON fato_vendas_base (-- resposta_aqui, resposta_aqui)
"""


In [95]:
query = """SELECT
    venda_sk,
    order_id,
    cliente_sk,
    produto_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000
  AND produto_sk = 15420
  """

banco_dados.sql(query).df()


,venda_sk,order_id,cliente_sk,produto_sk,valor_total
0,71606,a6dfab363a9e239287c4b1e3c275fb93,5000,15420,336.56
